# IDR Snowflake Connection Examples

This notebook provides some examples showing how to connect to the IDR Snowflake instance. Connecting to IDR Snowflake requires that you be connected to the CMS VPN as well as Snowflake-appropriate job codes. See [this Confluence page](https://confluenceent.cms.gov/display/IDRCC/IDRC+Onboarding+-+Snowflake+UI+Access) for instructions. 


## Browser-based connections (Snowsight)

One big advantage of Snowflake is that you can log in and manipulate data directly using a browser interface, which is called Snowsight. To get to the IDR Snowflake instance in the browser, navigate to   [https://app-cms-idr.privatelink.snowflakecomputing.com/cms/idr/](https://app-cms-idr.privatelink.snowflakecomputing.com/cms/idr/), and authenticate with your EUA credentials. And, that's it! At this point, you can click around, manipulate/explore data, and do whatever else you want.

## Python

You can use the same connection your browser uses to fetch data and run queries in Snowflake using Python. To run queries in the IDR Snowflake instance in Python, you'll need to use a Python connector library. There are two main options I'd recommend:
- Pandas-native Snowflake interface using [base Pandas](https://pandas.pydata.org/) and [Snowflake connector](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector)
- [Snowflake-native Pandas interface](https://docs.snowflake.com/en/developer-guide/python-connector/python-connector-install)

Personally, I find the Pandas-native interface a little more natural and succinct, but if you're not used to Pandas coding patterns, you might prefer the Snowflake-native option. Either one is fine, though.

Another approach you can use is [Snowflake SQLAlchemy](https://docs.snowflake.com/en/developer-guide/python-connector/sqlalchemy). However, I wouldn't recommend the SQLAlchemy option unless you're working in a software development environment. Using the SQLAlchemy framework directly has some advantages for software development, but it's unnecessary overhead for running individual queries or small-scale data analysis projects.

## Pandas-native Snowflake interface

In [ ]:
# Install Pandas and the base Snowflake connector
# This is only necessary the first time you build a connection, though it won't hurt you to run it again
# You may need to restart your notebook to access newly-installed packages

%pip install "pandas"
%pip install "snowflake-connector-python"

In [ ]:
# Import Pandas and the Snowflake connector

# Make sure to fill in your EUA below!
# Using the "externalbrowser" authentication option will open a browser window, where you'll be prompted to authenticate
# This connection will be live until approximately 10 minutes of inactivity, after which you'll need to reauthenticate

import pandas as pd

import snowflake.connector

con = snowflake.connector.connect(
    user="<your_eua>",
    account="cms-idr.privatelink",
    authenticator="externalbrowser"
)


In [ ]:
# Now, you can run any query you like!
# For example, count of living Medicare beneficiaries by state and zip
# We'll define the query, then use the Pandas .read_sql() method to run the query using our connection from above
# This may output a Pandas warning regarding SQLAlchemy, but this is something you can ignore

query = """
SELECT GEO_USPS_STATE_CD AS STATE
    , GEO_ZIP5_CD AS ZIP5
    , COUNT(DISTINCT BENE_SK) AS COUNT
FROM IDRC_PRD.CMS_VDM_VIEW_MDCR_PRD.V2_MDCR_BENE
WHERE BENE_DEATH_DT IS NULL OR BENE_DEATH_DT >= CURRENT_DATE()
GROUP BY STATE, GEO_ZIP5_CD
ORDER BY STATE, GEO_ZIP5_CD
"""

result = pd.read_sql(query, con)

result

In [ ]:
# From here, you can can manipulate data in Pandas, save to a CSV, or whatever other format you like
result.to_csv('~/Desktop/medicare_bene_counts.csv', index=False)

## Snowflake-native Pandas interface

In [ ]:
# Install the Snowflake Pandas connector
# This is only necessary the first time you build a connection, though it won't hurt you to run it again
# You may need to restart your notebook to access newly-installed packages

%pip install "snowflake-connector-python[pandas]"

In [ ]:
# Import the library, and create a connector

# First step is the same as above
# Fill in your EUA, create a connection, authenticate in the browser

import snowflake.connector

con = snowflake.connector.connect(
    user="<your_eua>",
    account="cms-idr.privatelink",
    authenticator="externalbrowser"
)


In [ ]:
# Now, you can run any query you like!
# As before, count of living Medicare beneficiaries by state and zip
# We'll define the query, then create a "cursor" that we can use to actually run queries
# and, use the .fetch_pandas_all() method to fetch the results as a Pandas dataframe
# after that, everything is the same as above

query = """
SELECT GEO_USPS_STATE_CD AS STATE
    , GEO_ZIP5_CD AS ZIP5
    , COUNT(DISTINCT BENE_SK) AS COUNT
FROM IDRC_PRD.CMS_VDM_VIEW_MDCR_PRD.V2_MDCR_BENE
WHERE BENE_DEATH_DT IS NULL OR BENE_DEATH_DT >= CURRENT_DATE()
GROUP BY STATE, GEO_ZIP5_CD
ORDER BY STATE, GEO_ZIP5_CD
"""

cur = con.cursor()
result = cur.execute(query).fetch_pandas_all()

result

## Other languages

You can also authenticate to Snowflake using pretty much any other language you like. I'm less familiar with these options, but here are a few resources to get you started. All of these options should work as long as you're in an environment with networking permissions to access IDR Snowflake (i.e. you can resolve the Snowflake browser link above and successfully authenticate).
 - [SAS ODBC instructions](https://community.snowflake.com/s/article/How-to-setup-Snowflake-connectivity-with-SAS)
 - [R ODBC instructions](https://community.snowflake.com/s/article/How-To-Connect-Snowflake-with-R-RStudio-using-RODBC-driver-on-Windows-MacOS-Linux)